In [ ]:
! pip install torch pandas scikit-learn


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# load training and testing datasets from CSV files
trainDf=pd.read_csv("data/train_clean.csv")
testDf=pd.read_csv("data/test_clean.csv")

# separate features (all columns except last) and target (last column)
xTrain=trainDf.iloc[:,:-1].values
yTrain=trainDf.iloc[:,-1].values.reshape(-1,1)
xTest=testDf.iloc[:,:-1].values
yTest=testDf.iloc[:,-1].values.reshape(-1,1)

# normalize feature values so neural network trains more smoothly
scaler=StandardScaler()
xTrain=scaler.fit_transform(xTrain)
xTest=scaler.transform(xTest)

# convert numpy arrays into PyTorch tensors
xTrain=torch.tensor(xTrain,dtype=torch.float32)
yTrain=torch.tensor(yTrain,dtype=torch.float32)
xTest=torch.tensor(xTest,dtype=torch.float32)
yTest=torch.tensor(yTest,dtype=torch.float32)

# define neural network architecture
# input size = number of features
# hidden layer with 64 neurons using ReLU activation
# output layer produces single value (regression)
model=nn.Sequential(
    nn.Linear(xTrain.shape[1],64),
    nn.ReLU(),
    nn.Linear(64,1)
)

# mean squared error loss for regression tasks
lossFn=nn.MSELoss()

# Adam optimizer adjusts weights using gradient descent
optimizer=optim.Adam(model.parameters(),lr=0.001)

# training loop
for epoch in range(100):
    pred=model(xTrain)              # forward pass (predictions)
    loss=lossFn(pred,yTrain)        # calculate error
    
    optimizer.zero_grad()           # clear previous gradients
    loss.backward()                 # backpropagation
    optimizer.step()                # update weights

# evaluation on test dataset without gradient tracking
with torch.no_grad():
    testPred=model(xTest)
    testLoss=lossFn(testPred,yTest)

# print final test loss value
print(testLoss.item())
